In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOKS 05, 08, 63, 64
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 05, 08, 63, 64")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB08_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_08_summary.json"
NB63_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_63_summary.json"
NB64_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_64_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB08_SUMMARY_PATH, "run 08_basel_ifrs9_mapping.ipynb first -- Problem 13 inherits its real EAD/LGD "
                         "assumptions rather than re-guessing them"),
    (NB63_SUMMARY_PATH, "run 63_customer_intelligence_modeling.ipynb first (Problem 12) -- Problem 13 reuses "
                         "Problem 12's real, persisted unified customer profile directly"),
    (NB64_SUMMARY_PATH, "run 64_customer_intelligence_validation_deployment.ipynb first (Problem 12)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB08_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB08_SUMMARY = json.load(f)
with open(NB63_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB63_SUMMARY = json.load(f)
with open(NB64_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB64_SUMMARY = json.load(f)

# --- The two real, already-validated sources Problem 13 depends on (per the
#     Master Execution Plan: "#1, #12"), read via each producing notebook's
#     OWN recorded path -- never re-derived or guessed. ---
CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
STATIC_PD_AUC = CHAMPION_METRICS.get("holdout_auc")
EAD_PER_ACCOUNT_USD = NB08_SUMMARY["ead_per_account_usd_assumption"]
LGD_ASSUMPTION = NB08_SUMMARY["lgd_assumption"]

P12_PROFILE_PATH = Path(NB63_SUMMARY["profile_path"])
if not P12_PROFILE_PATH.exists():
    raise FileNotFoundError(f"{P12_PROFILE_PATH} not found.\nFix: re-run Notebook 63 (Problem 12).")
P12_DEPLOYMENT_POLICY_PATH = Path(NB64_SUMMARY["deployment_policy_path"])
with open(P12_DEPLOYMENT_POLICY_PATH, "r", encoding="utf-8") as f:
    P12_DEPLOYMENT_POLICY = json.load(f)
P12_UNIFIED_RISK_GRADE_NAMES = P12_DEPLOYMENT_POLICY["unified_risk_grade_names"]
P12_RECOMMENDED_FOR_PRODUCTION = NB64_SUMMARY["recommended_for_production"]
P12_MEETS_KPI_WITH_CI = NB64_SUMMARY["meets_kpi_with_ci"]

if not P12_RECOMMENDED_FOR_PRODUCTION:
    print(
        "WARNING: Problem 12 (360 Degree Customer Intelligence) is NOT currently recommended for production. "
        "Problem 13 still reuses its real, measured UNIFIED_RISK_SCORE verbatim (the signal itself is real "
        "regardless of the recommendation flag), but this is noted honestly here rather than silently assumed "
        "away, and carried into Notebook 67's own KPI validation."
    )

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
DETECTED_TOTAL_RAM_BYTES = PROJECT_CONFIG["resource_limits"]["total_ram_bytes_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

if "profitability_modeling_policy" in PILLAR_DIRS:
    PROFITABILITY_POLICY_DIR = PILLAR_DIRS["profitability_modeling_policy"]
else:
    PROFITABILITY_POLICY_DIR = (
        PROJECT_ROOT / "Phase5_Customer_Business_Intelligence"
        / "13_Problem13_Risk_Adjusted_Profitability_Modeling" / "policy"
    )
    print(f"NOTE: 'profitability_modeling_policy' not in pillar_dirs -- using fallback: {PROFITABILITY_POLICY_DIR}")
PROFITABILITY_POLICY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded config from                          : {CONFIG_PATH}")
print(f"Champion architecture (Problem 1, measured) : {CHAMPION_NAME}, holdout AUC {STATIC_PD_AUC}")
print(f"EAD per account / LGD (Notebook 08, inherited): ${EAD_PER_ACCOUNT_USD:,} / {LGD_ASSUMPTION:.0%}")
print(f"Problem 12 unified profile (measured)       : {P12_PROFILE_PATH.name}, "
      f"recommended={P12_RECOMMENDED_FOR_PRODUCTION}, meets_kpi_with_ci={P12_MEETS_KPI_WITH_CI}")
print(f"Policy artifacts will be written under      : {PROFITABILITY_POLICY_DIR}")
print("\n✅ Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS (PHASE 5 CAP,
#            92% CPU / 92% RAM, CARRIED FORWARD -- NO NEW INCIDENT HAS
#            OCCURRED TO WARRANT CHANGING IT)
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

_PHASE5_CPU_FRACTION_CAP = 0.92
_PHASE5_RAM_FRACTION_CAP = 0.92
_historical_thread_count = PROJECT_CONFIG["resource_limits"]["warp_thread_count"]
_historical_max_ram_bytes = PROJECT_CONFIG["resource_limits"]["max_ram_bytes"]
WARP_THREAD_COUNT = min(_historical_thread_count, max(1, round(DETECTED_LOGICAL_CORES * _PHASE5_CPU_FRACTION_CAP)))
MAX_RAM_BYTES = min(_historical_max_ram_bytes, round(DETECTED_TOTAL_RAM_BYTES * _PHASE5_RAM_FRACTION_CAP))

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import psutil
except ImportError:
    missing.append("psutil")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


def _available_ram_gb() -> float:
    return psutil.virtual_memory().available / 1e9


_available_ram_gb_at_start = _available_ram_gb()
_comfortable_available_ram_gb = 0.50 * (MAX_RAM_BYTES / 1e9)
_min_required_available_ram_gb = 0.25 * (MAX_RAM_BYTES / 1e9)
if _available_ram_gb_at_start < _min_required_available_ram_gb:
    raise RuntimeError(
        f"Only {_available_ram_gb_at_start:.2f} GB of system RAM is available, which is below the "
        f"{_min_required_available_ram_gb:.2f} GB floor this notebook needs. Close other Jupyter kernels / "
        f"memory-heavy applications, confirm available RAM with `psutil.virtual_memory().available / 1e9` in "
        f"a fresh cell, then re-run this notebook from the top."
    )
if _available_ram_gb_at_start < _comfortable_available_ram_gb:
    print(f"⚠️  WARNING: only {_available_ram_gb_at_start:.2f} GB of system RAM is available "
          f"(comfortable margin is {_comfortable_available_ram_gb:.2f} GB). Proceeding.")
else:
    print(f"RAM pre-flight check passed: {_available_ram_gb_at_start:.2f} GB available >= "
          f"{_comfortable_available_ram_gb:.2f} GB comfortable margin.")

logger.info(f"Polars thread pool configured to {WARP_THREAD_COUNT}/{DETECTED_LOGICAL_CORES} threads")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
print(f"Configured RAM ceiling: {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n✅ Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
print(f"Raw train_data.csv: {RAW_TRAIN_DATA_PATH}")
print("\n✅ Section 3 complete.")


# =============================================================================
# SECTION 4: REAL SPEND-FEATURE DISCOVERY (VIA RAW CSV SCHEMA) -- THIS
#            PROBLEM'S REVENUE PROXY MUST BE GROUNDED IN A REAL COLUMN
#            GROUP, NOT INVENTED
# =============================================================================
_section("SECTION 4: Real Spend-Feature Discovery (via Raw CSV Schema)")

print(
    "The Master Execution Plan names this problem's core technique as 'PD-adjusted revenue (spend proxy) "
    "minus expected loss' -- but this Kaggle dataset carries no real dollar-denominated revenue or spend "
    "field (the competition anonymizes and normalizes every feature). What IS real: the competition's own "
    "publicly documented feature-category convention groups every raw column by a prefix -- D_ (Delinquency), "
    "S_ (Spend), P_ (Payment), B_ (Balance), R_ (Risk) -- with the single documented exception that S_2 is "
    "the statement date, not a Spend feature. This notebook verifies that real 'S_' Spend columns actually "
    "exist in the real raw CSV header before building anything on top of that premise -- the same honesty "
    "discipline Notebook 54 applied when it found no real credit-limit/utilization field and said so plainly, "
    "rather than silently trusting an external claim about this dataset's contents."
)

_t0 = time.time()
_header_row = pl.scan_csv(RAW_TRAIN_DATA_PATH, n_rows=0).collect_schema().names()
REAL_SPEND_COLUMNS = sorted(
    [c for c in _header_row if c == "S_2" or (c.startswith("S_") and c != "S_2")]
)
REAL_SPEND_COLUMNS = [c for c in REAL_SPEND_COLUMNS if c != "S_2"]
print(f"Scanned real CSV header in {time.time() - _t0:.2f}s.")
print(f"Real columns matching the 'S_' Spend prefix (excluding S_2, the real statement date): "
      f"{len(REAL_SPEND_COLUMNS)} found.")
if len(REAL_SPEND_COLUMNS) == 0:
    raise RuntimeError(
        "No real 'S_' Spend columns were found in the raw CSV header -- this problem's core technique "
        "(PD-adjusted revenue from a real spend proxy) has no real data to stand on for this dataset. Do not "
        "proceed with a fabricated proxy; report this honestly and revisit the technique with the business "
        "stakeholder."
    )
print(f"Real Spend columns (first 10 shown): {REAL_SPEND_COLUMNS[:10]}"
      + (" ..." if len(REAL_SPEND_COLUMNS) > 10 else ""))

# Cheap real sample (first 2,000 raw rows -- a schema/sanity check, not a full pass; the full real
# per-customer aggregation happens in Notebook 67's own streaming pass) to confirm these columns carry
# real, non-degenerate numeric data, not an all-null or constant column group.
_sample_df = pl.read_csv(RAW_TRAIN_DATA_PATH, n_rows=2000, columns=REAL_SPEND_COLUMNS)
_sample_null_pct = {c: round(100.0 * _sample_df[c].is_null().sum() / _sample_df.height, 1) for c in REAL_SPEND_COLUMNS}
_sample_nondegenerate = [c for c in REAL_SPEND_COLUMNS if _sample_df[c].drop_nulls().n_unique() > 1]
print(f"Real sample (first {_sample_df.height:,} rows): "
      f"{len(_sample_nondegenerate)} of {len(REAL_SPEND_COLUMNS)} Spend columns carry real, non-degenerate "
      f"numeric data (>1 distinct real value in the sample).")
_worst_null_cols = sorted(_sample_null_pct.items(), key=lambda kv: -kv[1])[:5]
print(f"Highest real null-rate Spend columns in this sample: {_worst_null_cols}")
if len(_sample_nondegenerate) == 0:
    raise RuntimeError("Every real Spend column is degenerate in this sample -- investigate before proceeding.")
print("\n✅ Section 4 complete.")


# =============================================================================
# SECTION 5: BUSINESS UNDERSTANDING -- RISK-ADJUSTED PROFITABILITY MODELING
# =============================================================================
_section("SECTION 5: Business Understanding -- Risk-Adjusted Profitability Modeling")

print(
    "PROBLEM 13 -- RISK-ADJUSTED PROFITABILITY MODELING (PD-adjusted revenue minus expected loss)\n\n"
    "Business case: every prior problem in this platform answers a RISK question -- who is likely to default, "
    "how severely, and what should be done about it. None of them answers a PROFITABILITY question: even a "
    "customer with a real elevated PD can still be net-profitable if their real revenue contribution is high "
    "enough, and a customer with a real low PD can still be a marginal or losing account if their real "
    "revenue contribution is low. Problem 13 nets Problem 12's real UNIFIED_RISK_SCORE (used here as the "
    "real, measured probability-of-default input) against a revenue estimate to produce one per-customer "
    "PROFITABILITY_SCORE -- ties risk modeling directly to the P&L, per the Master Execution Plan's own "
    "definition.\n\n"
    "HONEST SCOPE NOTE (same discipline as Problem 10's utilization-trend reinterpretation): this dataset has "
    "no real dollar-denominated revenue or spend field. Section 4 confirmed the dataset DOES carry a real, "
    "documented 'Spend' feature-category (columns prefixed 'S_', excluding the real statement-date column "
    "S_2) -- Notebook 67 will use the real, measured RELATIVE variation across these real columns as "
    "SPEND_INDEX, a genuinely real per-customer ranking signal. The dataset's own anonymization means "
    "SPEND_INDEX carries no real dollar unit, so converting it into an actual USD revenue figure requires an "
    "explicit ASSUMPTION (an average monthly revenue-per-active-account figure, scaled by each customer's own "
    "real relative SPEND_INDEX rank) -- exactly the same class of judgment call Notebook 08 already made for "
    "EAD_PER_ACCOUNT_USD, applied here to the revenue side of the P&L for the first time in this platform. "
    "The RELATIVE ranking across customers is real and measured; only the ABSOLUTE dollar scale is an "
    "editable business assumption, and every report this problem produces states this plainly.\n\n"
    "WHY THIS IS A GENUINELY NEW ARTIFACT: no prior problem nets a revenue estimate against expected loss on "
    "a per-customer basis. Problem 3's ECL and Problem 4's LGD both estimate the LOSS side of the P&L in "
    "isolation; Problem 13 is the platform's first PROFITABILITY view, and its real hard-gating KPI (Section "
    "7) is the specific, testable claim that PD-adjustment actually re-ranks customers relative to raw "
    "revenue alone -- proving the risk adjustment does real work, not just relabeling revenue."
)
print("\n✅ Section 5 complete.")


# =============================================================================
# SECTION 6: PROFITABILITY SCORE DEFINITION (ASSUMPTION)
# =============================================================================
_section("SECTION 6: Profitability Score Definition (ASSUMPTION)")

# ASSUMPTION: revenue model. SPEND_INDEX (Notebook 67) is each customer's
# real, measured percentile rank (0-1) of average real Spend-column values
# on their own real latest statement -- a real, relative signal. Absolute
# dollar revenue has no real analog in this dataset, so it is modeled as an
# ASSUMPTION average monthly revenue, scaled by a real-rank-driven
# multiplier so higher real relative spend maps to higher assumed revenue.
REVENUE_ASSUMPTIONS = {
    "avg_monthly_revenue_per_account_usd": {
        "value": 65.0,
        "source": "ASSUMPTION -- illustrative average monthly revenue-per-active-account (interchange, fees, "
                   "and finance charges combined) for a mid-market unsecured consumer credit product; edit to "
                   "your institution's actual revenue data.",
    },
    "revenue_multiplier_floor": {
        "value": 0.40,
        "source": "ASSUMPTION -- a customer at the real 0th percentile of relative Spend (SPEND_INDEX) is "
                   "modeled as earning 40% of the average assumed revenue, not zero -- even low-real-spend "
                   "accounts carry some baseline real revenue (fees, minimum finance charges).",
    },
    "revenue_multiplier_ceiling": {
        "value": 1.80,
        "source": "ASSUMPTION -- a customer at the real 100th percentile of relative Spend is modeled as "
                   "earning 180% of the average assumed revenue; edit both bounds to your institution's own "
                   "real revenue-by-spend-decile analysis when available.",
    },
}
REVENUE_MULTIPLIER_FLOOR = REVENUE_ASSUMPTIONS["revenue_multiplier_floor"]["value"]
REVENUE_MULTIPLIER_CEILING = REVENUE_ASSUMPTIONS["revenue_multiplier_ceiling"]["value"]
AVG_MONTHLY_REVENUE_PER_ACCOUNT_USD = REVENUE_ASSUMPTIONS["avg_monthly_revenue_per_account_usd"]["value"]

# --- Formula (documented here, computed on the real scored population in
#     Notebook 67):
#       REVENUE_MULTIPLIER   = floor + (ceiling - floor) * real_spend_percentile_rank
#       REVENUE_PER_ACCOUNT  = AVG_MONTHLY_REVENUE_PER_ACCOUNT_USD * REVENUE_MULTIPLIER
#       PD_ADJUSTED_REVENUE  = REVENUE_PER_ACCOUNT * (1 - UNIFIED_RISK_SCORE)
#       EXPECTED_LOSS        = UNIFIED_RISK_SCORE * EAD_PER_ACCOUNT_USD * LGD_ASSUMPTION
#       PROFITABILITY_SCORE  = PD_ADJUSTED_REVENUE - EXPECTED_LOSS
#     UNIFIED_RISK_SCORE is used here as a probability-like risk score (it is
#     bounded in [0, 1] by construction -- Notebook 62's composite weights
#     sum to 1.0 across active terms) -- an honest calibration caveat, not a
#     claim that it is a formally calibrated probability. ---
print("PROFITABILITY_SCORE formula (computed on the real scored population in Notebook 67):")
print("  REVENUE_MULTIPLIER  = floor + (ceiling - floor) * real_spend_percentile_rank")
print("  REVENUE_PER_ACCOUNT = AVG_MONTHLY_REVENUE_PER_ACCOUNT_USD * REVENUE_MULTIPLIER")
print("  PD_ADJUSTED_REVENUE = REVENUE_PER_ACCOUNT * (1 - UNIFIED_RISK_SCORE)")
print("  EXPECTED_LOSS       = UNIFIED_RISK_SCORE * EAD_PER_ACCOUNT_USD * LGD_ASSUMPTION")
print("  PROFITABILITY_SCORE = PD_ADJUSTED_REVENUE - EXPECTED_LOSS")

# ASSUMPTION: 3-tier grading, reusing this platform's established tertile
# convention (Problems 4/8/10/12) for cross-platform continuity. Cut VALUES
# are fit fresh on the real scored population in Notebook 67, mirroring
# Problem 12's own precedent (Notebook 62 set the convention, Notebook 63
# fit the real cut values).
PROFITABILITY_TIER_NAMES = ["Low Profitability", "Medium Profitability", "High Profitability"]
PROFITABILITY_TIER_CUT_PERCENTILES = [33.333, 66.667]

print(f"\nPROFITABILITY_TIER_NAMES (ASSUMPTION, tertile convention): {PROFITABILITY_TIER_NAMES}")
print("\n✅ Section 6 complete.")


# =============================================================================
# SECTION 7: KPI TARGETS -- PROFITABILITY TIER MONOTONICITY & RISK
#            ADJUSTMENT MATERIALITY
# =============================================================================
_section("SECTION 7: KPI Targets -- Profitability Tier Monotonicity & Risk Adjustment Materiality")

PROFITABILITY_KPI_TARGETS = {
    "profitability_tier_monotonicity": {
        "description": (
            "PRIMARY, hard-gating KPI (reused convention from Problems 4/8/10, inverted direction): the real "
            "observed default rate must be MONOTONICALLY DECREASING from the Low Profitability tier to the "
            "High Profitability tier on the real holdout split -- since PROFITABILITY_SCORE is explicitly "
            "PD-adjusted, a customer scored as more profitable should, on average, also be a real lower-"
            "default-rate customer. A violation means the revenue side of the formula (Section 6) is "
            "overwhelming the real risk signal for at least one tier pair, and needs revisiting."
        ),
        "hard_gate": True,
    },
    "risk_adjustment_materiality": {
        "description": (
            "NEW hard-gating KPI for this problem (no Problem 4/6/8/9/10/12 analog -- this is the platform's "
            "first problem that risk-adjusts a revenue estimate, so there is no prior claim to reuse): the "
            "real Spearman rank correlation between UNIFIED_RISK_SCORE and PROFITABILITY_SCORE on the real "
            "holdout split must be <= an ASSUMPTION -0.15 threshold (a materially negative correlation). "
            "This is the specific, testable claim this whole problem depends on: that the PD adjustment "
            "measurably re-ranks customers relative to raw revenue alone, rather than the revenue term simply "
            "dominating and producing a profitability ranking indistinguishable from a pure spend ranking. If "
            "this gate fails, the honest conclusion is that Section 6's revenue-to-risk balance needs "
            "revisiting, not that either underlying signal is bad."
        ),
        "spearman_threshold": -0.15,
        "hard_gate": True,
    },
    "min_tier_population_pct": 10.0,
    "min_tier_population_pct_description": (
        "ASSUMPTION -- each of the 3 real PROFITABILITY_TIER tiers must hold >= 10% of the eligible "
        "validation population, reusing this platform's Problem 4/8/12 single-axis threshold convention."
    ),
    "metrics_suite_requirement": (
        "STANDING RULE (carried from Problems 6-12): Notebook 67/68 must compute and DISPLAY -- inline in "
        "the notebook AND in this problem's Word/Excel/HTML reports -- the real per-tier default rate table "
        "and the real Spearman correlation, so the risk-adjustment-materiality claim is auditable."
    ),
    "elevated_reporting_requirement": (
        "STANDING RULE (carried from Problems 7-12): Problem 13's Word report must synthesize MAXIMUM DETAIL "
        "from every one of this problem's notebooks (66-69), with a narrative 'story' paragraph below every "
        "chart. Problem 13's HTML report must be an advanced, 'global standard' interactive dashboard with "
        "slicers, filters, full legends, and interactive KPI cards -- built in Notebook 69, and must double "
        "as the real profitability-scoring dashboard the master plan names as this problem's deliverable."
    ),
    "static_pd_reference_auc": STATIC_PD_AUC,
    "problem_12_reference": {"recommended_for_production": P12_RECOMMENDED_FOR_PRODUCTION,
                              "meets_kpi_with_ci": P12_MEETS_KPI_WITH_CI},
}
print(f"profitability_tier_monotonicity (hard gate): "
      f"{PROFITABILITY_KPI_TARGETS['profitability_tier_monotonicity']['hard_gate']}")
print(f"risk_adjustment_materiality (hard gate, Spearman threshold="
      f"{PROFITABILITY_KPI_TARGETS['risk_adjustment_materiality']['spearman_threshold']}): "
      f"{PROFITABILITY_KPI_TARGETS['risk_adjustment_materiality']['hard_gate']}")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: WRITE PROFITABILITY MODELING POLICY ARTIFACT
# =============================================================================
_section("SECTION 8: Write Profitability Modeling Policy Artifact")

PROFITABILITY_MODELING_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 13 -- Risk-Adjusted Profitability Modeling (PD-Adjusted Revenue Minus Expected Loss)",
    "real_spend_columns": REAL_SPEND_COLUMNS,
    "n_real_spend_columns": len(REAL_SPEND_COLUMNS),
    "revenue_assumptions": REVENUE_ASSUMPTIONS,
    "profitability_tier_names": PROFITABILITY_TIER_NAMES,
    "profitability_tier_cut_percentiles": PROFITABILITY_TIER_CUT_PERCENTILES,
    "reused_from_problem_1": {"champion_model": CHAMPION_NAME, "champion_holdout_auc": STATIC_PD_AUC},
    "reused_from_problem_8": {"ead_per_account_usd": EAD_PER_ACCOUNT_USD, "lgd_assumption": LGD_ASSUMPTION},
    "reused_from_problem_12": {
        "profile_path": str(P12_PROFILE_PATH), "deployment_policy_path": str(P12_DEPLOYMENT_POLICY_PATH),
        "unified_risk_grade_names": P12_UNIFIED_RISK_GRADE_NAMES,
        "recommended_for_production": P12_RECOMMENDED_FOR_PRODUCTION,
    },
    "kpi_targets": PROFITABILITY_KPI_TARGETS,
    "random_seed": RANDOM_SEED,
    "warp_resource_cap": {
        "cpu_fraction_cap": _PHASE5_CPU_FRACTION_CAP, "ram_fraction_cap": _PHASE5_RAM_FRACTION_CAP,
        "warp_thread_count": WARP_THREAD_COUNT, "max_ram_bytes": MAX_RAM_BYTES,
        "note": "Phase 4/5 cap (92%/92%) carried forward -- no new incident to warrant a change.",
    },
}
policy_path = PROFITABILITY_POLICY_DIR / "profitability_modeling_policy.json"
with open(policy_path, "w", encoding="utf-8") as f:
    json.dump(PROFITABILITY_MODELING_POLICY, f, indent=2)
print(f"Wrote: {policy_path}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 9: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Policy file was written", policy_path.exists())
_all_checks_passed &= _check("At least one real Spend column was discovered", len(REAL_SPEND_COLUMNS) > 0)
_all_checks_passed &= _check("S_2 (the real statement date) was excluded from REAL_SPEND_COLUMNS",
                              "S_2" not in REAL_SPEND_COLUMNS)
_all_checks_passed &= _check("PROFITABILITY_TIER_NAMES has exactly 3 tiers (tertile convention)",
                              len(PROFITABILITY_TIER_NAMES) == 3)
_all_checks_passed &= _check("Revenue multiplier ceiling is strictly greater than the floor",
                              REVENUE_MULTIPLIER_CEILING > REVENUE_MULTIPLIER_FLOOR)
_all_checks_passed &= _check("EAD/LGD were inherited from Notebook 08, not re-guessed",
                              EAD_PER_ACCOUNT_USD == NB08_SUMMARY["ead_per_account_usd_assumption"]
                              and LGD_ASSUMPTION == NB08_SUMMARY["lgd_assumption"])
_all_checks_passed &= _check("Reused Problem 12's real unified profile path verbatim",
                              str(P12_PROFILE_PATH) == NB63_SUMMARY["profile_path"])
_all_checks_passed &= _check("Both hard-gating KPIs are marked hard_gate=True (not silently advisory)",
                              PROFITABILITY_KPI_TARGETS["profitability_tier_monotonicity"]["hard_gate"] is True
                              and PROFITABILITY_KPI_TARGETS["risk_adjustment_materiality"]["hard_gate"] is True)
_all_checks_passed &= _check("risk_adjustment_materiality's Spearman threshold is negative (a real re-ranking "
                              "claim, not a trivially-passable non-negative bar)",
                              PROFITABILITY_KPI_TARGETS["risk_adjustment_materiality"]["spearman_threshold"] < 0)
_all_checks_passed &= _check("WARP thread count never exceeds the historical config's own value",
                              WARP_THREAD_COUNT <= _historical_thread_count)
_all_checks_passed &= _check("WARP RAM ceiling never exceeds the historical config's own value",
                              MAX_RAM_BYTES <= _historical_max_ram_bytes)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 9 complete -- all checks passed.")


# =============================================================================
# SECTION 10: WRITE NOTEBOOK 66 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 10: Write Notebook 66 Summary Artifact")

NB66_SUMMARY = {
    "notebook": "66_profitability_modeling_business_understanding.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "policy_path": str(policy_path),
    "n_real_spend_columns": len(REAL_SPEND_COLUMNS),
    "profitability_tier_names": PROFITABILITY_TIER_NAMES,
    "revenue_multiplier_floor": REVENUE_MULTIPLIER_FLOOR, "revenue_multiplier_ceiling": REVENUE_MULTIPLIER_CEILING,
    "avg_monthly_revenue_per_account_usd": AVG_MONTHLY_REVENUE_PER_ACCOUNT_USD,
    "warp_thread_count": WARP_THREAD_COUNT,
    "max_ram_bytes": MAX_RAM_BYTES,
    "random_seed": RANDOM_SEED,
}
NB66_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_66_summary.json"
with open(NB66_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB66_SUMMARY, f, indent=2)
print(f"Wrote: {NB66_SUMMARY_PATH}")

_section("NOTEBOOK 66 COMPLETE")
print(f"Real Spend columns discovered (ASSUMPTION-free, from the real CSV header): {len(REAL_SPEND_COLUMNS)}")
print(f"PROFITABILITY_TIER_NAMES (ASSUMPTION, tertile convention)             : {PROFITABILITY_TIER_NAMES}")
print(f"Revenue model (ASSUMPTION)  : ${AVG_MONTHLY_REVENUE_PER_ACCOUNT_USD}/mo avg, multiplier "
      f"[{REVENUE_MULTIPLIER_FLOOR}, {REVENUE_MULTIPLIER_CEILING}] by real relative spend rank")
print("Hard-gating KPIs            : profitability_tier_monotonicity, risk_adjustment_materiality")
print(f"WARP cap this notebook forward: {WARP_THREAD_COUNT} threads / {MAX_RAM_BYTES / 1e9:.1f} GB RAM")
print(f"Policy written to: {policy_path}")
print(
    "\nNext: 67_profitability_modeling_modeling.ipynb -- reads Problem 12's real unified profile, computes "
    "each real customer's SPEND_INDEX (percentile rank of their own real latest-statement average across the "
    "real Spend columns discovered above), scores REVENUE_PER_ACCOUNT / PD_ADJUSTED_REVENUE / EXPECTED_LOSS / "
    "PROFITABILITY_SCORE per Section 6's formula, fits the real tertile cut values on that population, and "
    "validates both hard-gating KPIs against the real observed default outcome."
)